# 03 — JUMP cpg0016 per-compound morphology vectors (B3b primary)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cellduet/blob/main/notebooks/03_jump_percompound.ipynb)

Pulls the harmony-corrected, MAD-int-feature-selected CellProfiler features for the JUMP-Cell Painting Consortium compound arm (`cpg0016-jump`), filters to the **228 Tahoe-overlap InChIKeys** identified in notebook 01, and aggregates per compound by mean across replicate wells. Output: **228 × 737-d** matrix, the headline morphology product for the B3b primary arm.

Uses pyarrow predicate pushdown over the S3 parquet so we never materialise the full 803,853-row table; only the ~1,500 wells matching our target compounds get read into memory.

| Step | Source | Size |
|---|---|---|
| Compound manifest | HF `patrickjreed/cellduet-compound-manifest` | 36 KB |
| InChIKey ↔ JCP2022 map | GitHub `jump-cellpainting/datasets/metadata/compound.csv.gz` | ~3 MB |
| Harmony parquet | S3 `cellpainting-gallery/cpg0016-jump-assembled/...` | 2.64 GB total, ~5 MB read after filter |
| Output | HF `patrickjreed/cellduet-jump-percompound` | ~700 KB |

Refs: `docs/datasets/jump-cp-cpg0016.md`, `docs/datasets/joint.md`.

## 1. Install + imports

In [ ]:
!pip install -q pandas pyarrow s3fs huggingface_hub
!pip install -q --no-deps "git+https://github.com/PatrickJReed/cellduet.git@main"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from huggingface_hub import HfApi, hf_hub_download, login

from cellduet.jump import aggregate_per_compound

print("imports OK")

## 2. HF login + cache

In [ ]:
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)
    print("HF login: OK (Colab secret)")
except Exception as e:
    print(f"Colab secret not used ({type(e).__name__}); relying on local cache or anon HF reads")

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE = Path("/content/drive/MyDrive/cellduet/cache")
except (ImportError, Exception) as e:
    print(f"Drive skipped ({type(e).__name__}); using local cache")
    CACHE = Path.home() / ".cache" / "cellduet"
CACHE.mkdir(parents=True, exist_ok=True)
print(f"cache: {CACHE}")

## 3. Pull manifest + get B3b target InChIKeys

The 228 InChIKeys that intersect Tahoe with JUMP cpg0016, computed in notebook 01.

In [ ]:
manifest_path = hf_hub_download(
    "patrickjreed/cellduet-compound-manifest", "compound_manifest.parquet", repo_type="dataset"
)
manifest = pd.read_parquet(manifest_path)
b3b_keys = set(manifest.loc[manifest["in_jump_cpg0016"], "inchikey_full"].dropna())
print(f"B3b target InChIKeys: {len(b3b_keys)}")

## 4. JUMP compound metadata → JCP2022 IDs

The harmony parquet keys rows on `Metadata_JCP2022`, not InChIKey. The JUMP-CP datasets repo on GitHub ships the compound table that maps the two.

In [ ]:
jump_url = "https://github.com/jump-cellpainting/datasets/raw/main/metadata/compound.csv.gz"
jump = pd.read_csv(jump_url)
jcp_map = jump[jump["Metadata_InChIKey"].isin(b3b_keys)][
    ["Metadata_JCP2022", "Metadata_InChIKey"]
].copy()
target_jcps = set(jcp_map["Metadata_JCP2022"])
print(
    f"JCP2022 IDs for the {len(b3b_keys)} target InChIKeys: {len(target_jcps)} "
    f"({len(jcp_map)} rows; one InChIKey can have multiple JCP2022 entries for stereoisomers)"
)

## 5. Download harmony parquet from S3 and filter locally

The full parquet is 2.84 GB on disk. We download once (cached to Drive on Colab) and use pyarrow's filter pushdown on the local file to extract just the rows whose `Metadata_JCP2022` is in our 228-compound target set. Result is a few thousand rows × 741 columns (4 metadata + 737 features).

Downloading from S3 first is faster + more deterministic than reading the parquet via remote `s3fs`, because this file has a single row group and pyarrow cannot do row-level filter pushdown across a remote read.

In [ ]:
import time

fs = s3fs.S3FileSystem(anon=True)
JUMP_S3 = (
    "cellpainting-gallery/cpg0016-jump-assembled/source_all/workspace/"
    "profiles_assembled/COMPOUND/v1.0/profiles_var_mad_int_featselect_harmony.parquet"
)
local_parquet = CACHE / "jump_cpg0016_harmony.parquet"

# Download once, ~2.84 GB. Pyarrow over remote s3fs is unreliable here because
# the parquet has a single row group and no row-level pushdown is possible;
# downloading first and filtering locally is faster + deterministic.
if not local_parquet.exists():
    print(f"downloading 2.84 GB harmony parquet to {local_parquet}...")
    t0 = time.time()
    fs.get(JUMP_S3, str(local_parquet))
    print(f"  done in {time.time()-t0:.1f}s")
else:
    print(f"using cached parquet at {local_parquet}")

print(f"  size: {local_parquet.stat().st_size/1e9:.2f} GB")

print("\nfiltering rows by Metadata_JCP2022...")
t0 = time.time()
filter_expr = pa.compute.field("Metadata_JCP2022").isin(list(target_jcps))
table = pq.read_table(str(local_parquet), filters=filter_expr)
df = table.to_pandas()
print(f"  filtered shape: {df.shape}  in {time.time()-t0:.1f}s")
print(f"  unique JCP2022 ids in result: {df['Metadata_JCP2022'].nunique()}")

## 6. Per-compound replicate-count gate

Acceptance gate per spec: ≥80% of compounds should have ≥3 wells of harmonised features.

In [ ]:
rep_counts = df.groupby("Metadata_JCP2022").size()
print(f"replicate-count summary across {len(rep_counts)} compounds:")
print(rep_counts.describe())
fraction_ok = (rep_counts >= 3).mean()
print(f"\nfraction with >=3 wells: {fraction_ok*100:.1f}%  (gate: >=80%)")
if fraction_ok < 0.80:
    print("WARNING: acceptance gate not met. Check parquet load + filter logic.")

## 7. Identify feature columns and aggregate per InChIKey

Aggregate by mean across replicate wells, joining JCP2022 back to InChIKey first so the output is keyed on the cross-modal join key.

In [ ]:
feat_cols = [c for c in df.columns if c.startswith("X_")]
meta_cols = [c for c in df.columns if c.startswith("Metadata_")]
print(f"feature columns: {len(feat_cols)} (expect 737)")
print(f"metadata columns: {meta_cols}")

# Join InChIKey, aggregate
df = df.merge(jcp_map, on="Metadata_JCP2022", how="left")
jump_per_compound = aggregate_per_compound(df, feature_cols=feat_cols, compound_col="Metadata_InChIKey")
print(f"\njump_per_compound shape: {jump_per_compound.shape}  (expect ~228 x 737)")

## 8. Save + push to HF

In [ ]:
jump_per_compound.index.name = "inchikey_full"
jump_path = CACHE / "jump_per_compound.parquet"
jump_per_compound.reset_index().to_parquet(jump_path, index=False)
print(f"saved {jump_path}  ({jump_path.stat().st_size/1024:.1f} KB)")

try:
    api = HfApi()
    repo_id = "patrickjreed/cellduet-jump-percompound"
    api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)
    api.upload_file(
        path_or_fileobj=str(jump_path),
        path_in_repo=jump_path.name,
        repo_id=repo_id,
        repo_type="dataset",
    )
    print(f"pushed to HF: {repo_id}")
except Exception as e:
    print(f"HF push skipped ({type(e).__name__}): {e}")

## 9. Summary

In [ ]:
print("JUMP cpg0016 per-compound morphology build summary (B3b primary)")
print("-" * 60)
print(f"target InChIKeys (B3b):      {len(b3b_keys)}")
print(f"JCP2022 ids resolved:        {len(target_jcps)}")
print(f"JUMP wells loaded:           {len(df)}")
print(f"per-compound matrix:         {jump_per_compound.shape}")
print(f"replicate-count >=3 fraction: {fraction_ok*100:.1f}%")
print()
print("ready for downstream cosine-distance + Mantel-r against Tahoe-pooled (notebook 06).")